<center><h1>Wagner_Anton_HW3</h1></center>
<br>
<br>

Name: Anton Wagner
<br>
Github Username: AntonW1
<br>
USC ID: [redacted since public repo]

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import re
from sklearn.feature_selection import f_classif

Get the AReM Data Set

In [2]:
data_path = '../Homework 3, 4 Data/AReM'

data = {}
columns = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
failed_files = []

for action in os.listdir(data_path):
    file_path = data_path + '/' + action
    if os.path.isdir(file_path):
        for dataset_name in os.listdir(file_path):
            dataset_path = file_path + '/' + dataset_name
            dataset_number = int(re.findall(r'\d+', dataset_name)[0])
            try:
                dataset = pd.read_csv(dataset_path, names=columns, usecols=range(7), header=None, comment='#', sep=r'[,\s]+', engine='python')
                data[(action, dataset_number)] = dataset
            except pd.errors.ParserError:
                failed_files.append(dataset_path)

print(f"Files read successfully: {len(data)}")
print(failed_files)

Files read successfully: 88
[]


Cycling activity datasets 9 and 14 had an extra comma at the end of the last row. Bending2 activity dataset 4 is separated with spaces instead of commas.
Using the following parameters with pd.read_csv allowed me to cleanly read the datasets:

```dataset = pd.read_csv(dataset_path, names=columns, usecols=range(7), header=None, comment='#', sep=r'[,\s]+', engine='python')```


### (b) Test and Train Data

In [3]:
train_test_split = {}

for (action, dataset_number) in data:
    test_dataset_numbers = []
    if action in ['bending1', 'bending2']:
        test_dataset_numbers = [1, 2]
    else:
        test_dataset_numbers = [1, 2, 3]

    if dataset_number in test_dataset_numbers:
        train_test_split[(action, dataset_number)] = 'test'
    else:
        train_test_split[(action, dataset_number)] = 'train'


print(pd.Series(train_test_split).value_counts())

train    69
test     19
Name: count, dtype: int64


Test set

bending1: 1, 2

bending2: 1, 2

cycling: 1, 2, 3

lying: 1, 2, 3

sitting: 1, 2, 3

standing: 1, 2, 3

walking: 1, 2, 3


Total: 19

This adds up correctly. The train set is the remaining 69 (88-19).

## (c) Feature Extraction

#### i. Research

##### Common time-domain features
Minimum - lowest value

Maximum - highest value

Mean - average value

Median - middle value

1st Quartile - value greater than 25% of values

3rd Quartile - value greater than 75% of values

Standard Deviation - how spread the values are

Skewness - asymmetry of the distribution

Kurtosis - heaviness of the tails

Autocorrelation - correlation of the series to a lagged version

Partial Autocorrelation - correlation of the series to a lagged version after removing intermediate lags

Source: https://www.geeksforgeeks.org/data-analysis/feature-engineering-for-time-series-data-methods-and-applications/ for Skewness, Kurtosis, Autocorrelation, Partial Autocorrelation

#### ii. Extraction

In [4]:
rows = []

for (action, dataset_number), dataset in data.items():
    row = {'activity': action, 'file': dataset_number, 'split': train_test_split[(action, dataset_number)]}
    for column in columns[1:]:
        time_series = dataset[column]
        row['min_' + column] = time_series.min()
        row['max_' + column] = time_series.max()
        row['mean_' + column] = time_series.mean()
        row['median_' + column] = time_series.median()
        row['q1_' + column] = time_series.quantile(0.25)
        row['q3_' + column] = time_series.quantile(0.75)
        row['std_' + column] = time_series.std()
        
    rows.append(row)

features = pd.DataFrame(rows)
print(features.shape)
features.head(10)

(88, 45)


,activity,file,split,min_avg_rss12,max_avg_rss12,mean_avg_rss12,median_avg_rss12,q1_avg_rss12,q3_avg_rss12,std_avg_rss12,...,q1_avg_rss23,q3_avg_rss23,std_avg_rss23,min_var_rss23,max_var_rss23,mean_var_rss23,median_var_rss23,q1_var_rss23,q3_var_rss23,std_var_rss23
0,bending1,1,test,37.25,45.00,40.624792,40.50,39.2500,42.00,1.476967,...,33.0000,36.00,2.188449,0.0,1.92,0.570583,0.43,0.00,1.30,0.582915
1,bending1,2,test,38.00,45.67,42.812812,42.50,42.0000,43.67,1.435550,...,32.0000,34.50,1.995255,0.0,3.11,0.571083,0.43,0.00,1.30,0.601010
2,bending1,3,train,35.00,47.40,43.954500,44.33,43.0000,45.00,1.558835,...,35.3625,36.50,1.999604,0.0,1.79,0.493292,0.43,0.00,0.94,0.513506
3,bending1,4,train,33.00,47.75,42.179812,43.50,39.1500,45.00,3.670666,...,30.4575,36.33,3.849448,0.0,2.18,0.613521,0.50,0.00,1.00,0.524317
4,bending1,5,train,33.00,45.75,41.678063,41.75,41.3300,42.75,2.243490,...,28.4575,31.25,2.411026,0.0,1.79,0.383292,0.43,0.00,0.50,0.389164
5,bending1,6,train,37.00,48.00,43.454958,43.25,42.5000,45.00,1.386098,...,22.2500,24.00,2.488862,0.0,5.26,0.679646,0.50,0.43,0.87,0.622534
6,bending1,7,train,36.25,48.00,43.969125,44.50,43.3100,44.67,1.618364,...,20.5000,23.75,3.318301,0.0,2.96,0.555312,0.49,0.00,0.83,0.487826
7,bending2,1,test,12.75,51.00,24.562958,24.25,23.1875,26.50,3.737514,...,20.5000,27.00,3.693786,0.0,4.97,0.700188,0.50,0.43,0.87,0.693720
8,bending2,2,test,0.00,42.75,27.464604,28.00,25.5000,30.00,3.583582,...,15.0000,20.75,5.053642,0.0,6.76,1.122125,0.83,0.47,1.30,1.012342
9,bending2,3,train,21.00,50.00,32.586208,33.00,26.1875,34.50,6.238143,...,17.6700,23.50,5.032424,0.0,13.61,1.162042,0.83,0.47,1.30,1.332980


#### iii. Standard Deviation

In [5]:
feature_columns = [column for column in features.columns if column not in ['activity', 'file', 'split']]
feature_std = features[feature_columns].std()
feature_std

min_avg_rss12       9.569975
max_avg_rss12       4.394362
mean_avg_rss12      5.335718
median_avg_rss12    5.440054
q1_avg_rss12        6.153590
q3_avg_rss12        5.138925
std_avg_rss12       1.772153
min_var_rss12       0.000000
max_var_rss12       5.062729
mean_var_rss12      1.574164
median_var_rss12    1.412244
q1_var_rss12        0.946386
q3_var_rss12        2.125266
std_var_rss12       0.884105
min_avg_rss13       2.956462
max_avg_rss13       4.875137
mean_avg_rss13      4.008380
median_avg_rss13    4.036396
q1_avg_rss13        4.220658
q3_avg_rss13        4.171628
std_avg_rss13       0.946710
min_var_rss13       0.000000
max_var_rss13       2.183625
mean_var_rss13      1.166114
median_var_rss13    1.145586
q1_var_rss13        0.843620
q3_var_rss13        1.552504
std_var_rss13       0.458242
min_avg_rss23       6.124001
max_avg_rss23       5.741238
mean_avg_rss23      5.675593
median_avg_rss23    5.813782
q1_avg_rss23        6.096465
q3_avg_rss23        5.531720
std_avg_rss23 

In [6]:
from scipy.stats import bootstrap

values = features[feature_columns].values

result = bootstrap(
    (values,),
    lambda x, axis: np.std(x, axis=axis, ddof=1),
    axis=0,
    confidence_level=0.90,
    n_resamples=1000,
    method='percentile',
    random_state=552,
)

confidence_interval = pd.DataFrame({
    'std': feature_std.values,
    'confidence_lower': result.confidence_interval.low,
    'confidence_upper': result.confidence_interval.high,
}, index=feature_columns).round(4)

confidence_interval

,std,confidence_lower,confidence_upper
min_avg_rss12,9.5700,8.3007,10.7491
max_avg_rss12,4.3944,3.3321,5.2757
mean_avg_rss12,5.3357,4.7225,5.9090
median_avg_rss12,5.4401,4.8068,5.9939
q1_avg_rss12,6.1536,5.5783,6.6210
q3_avg_rss12,5.1389,4.3261,5.8915
std_avg_rss12,1.7722,1.5713,1.9500
min_var_rss12,0.0000,0.0000,0.0000
max_var_rss12,5.0627,4.6276,5.4025
mean_var_rss12,1.5742,1.4031,1.6994


###### Reference
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html

#### iv. Select Features

In [7]:
train = features[features['split'] == 'train']
usable_features = [column for column in feature_columns if train[column].std() > 0]

# get the f-score of each feature for separating the actions
f_scores, p_values = f_classif(train[usable_features], train['activity'])
f_score_table = pd.Series(f_scores, index=usable_features).sort_values(ascending=False)
f_score_table

mean_var_rss12      900.156388
q3_var_rss12        807.048897
median_var_rss13    800.794772
q3_var_rss23        664.775928
mean_var_rss13      651.760982
mean_var_rss23      581.452582
median_var_rss23    563.125395
q3_var_rss13        561.899251
median_var_rss12    477.013486
std_var_rss12       360.894819
q1_var_rss23        346.341817
q1_var_rss12        265.456461
q1_var_rss13        244.818606
std_var_rss23       133.391176
std_var_rss13       128.617323
max_var_rss12        47.727323
max_avg_rss23        36.952832
mean_avg_rss23       33.431719
max_var_rss23        32.960327
q3_avg_rss23         32.241742
max_avg_rss13        31.693500
max_var_rss13        29.334628
median_avg_rss23     28.637573
q3_avg_rss13         28.105868
mean_avg_rss13       26.364974
q1_avg_rss23         25.493860
min_avg_rss23        24.096125
min_avg_rss13        22.712355
q1_avg_rss12         21.384206
q1_avg_rss13         21.192371
median_avg_rss13     19.409747
median_avg_rss12     17.489288
mean_avg

In [8]:
metrics = ['min', 'max', 'mean', 'median', 'q1', 'q3', 'std']
metric_scores = {m: f_score_table[f_score_table.index.str.startswith(m + '_')].mean() for m in metrics}
pd.Series(metric_scores).round(1).sort_values(ascending=False)

mean      368.2
q3        350.4
median    317.7
q1        154.1
std       109.2
max        30.0
min        14.6
dtype: float64

I am selecting mean, 3rd quartile, as well as standard deviation as the most important time-domain features.
I used an ANOVA F-score test to investigate how well each of the features separate the activities and then averaged them across the 6 time series. Mean (368.2) and 3rd quartile (350.4) were selected as they have the highest scores. I also chose standard deviation becuase I believe the signal spread provides meaningful information that the other two features do not capture. Although median and 1st quartile had higher scores, they may be more correlated with mean and 3rd quartile and thus not provide as much extra insight.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression model would ve expected to have a lower training RSS on a linear true relationship. The linear model is the same as the cubic model but with B2 = 0 and B3 = 0. This means that the cubic can fit the training data just as well as the linear and it is also able to overfit to exact noise of the training data closer than a straight line could get.

### (b) Linear Test

The linear regression model would likely have a lower test RSS on a linear true relationship as it is better able to represent the trend. On the test set, the noise that cubic regression model curved based on would no longer reflect the data.

### (c) Not Linear Train

The cubic regression model would be expected to have a lower training RSS on the true curved relationship as it is able to bend to match the curves of the training data. Similar to part a, the cubic model can fit at least as well as the linear model but with the ability to capture a curvature in addition.

### (d) Not Linear Testing

It would depend on how close the true relationship is to being linear. It might be possible for the linear regression model to have a lower test RSS on the true curved relationship if the curve is close to a straight line. Otherwise, the cubic regression model would likely perform better.